# Prepare Storage for Knowledge Mining Documents

Run this notebook one cell at a time to create or reuse the Lab 03 storage account, create a `documents` container, and upload the brochure PDFs used by the Azure AI Search indexer.

This notebook is an optional path for the portal steps **Create a Storage Account** and **Upload Documents to Azure Storage**. It uses `DefaultAzureCredential`, so it works with your current Azure sign-in context from VS Code, Azure CLI, or another supported Azure identity source.

## 1. Before You Start

Make sure `workshop/.env` exists. From the repository root, create it with `Copy-Item workshop/.env.sample workshop/.env` in PowerShell if needed.

Required setting: `STORAGE_RESOURCE_GROUP`. Optional settings: `STORAGE_SUBSCRIPTION_ID`, `AZURE_SUBSCRIPTION_ID`, `STORAGE_LOCATION`, and `STORAGE_ACCOUNT_NAME`.

Your signed-in Azure account needs permission to create storage accounts and assign roles. Owner or User Access Administrator at the resource group or subscription scope is typically sufficient.

In [1]:
import base64
import json
import os
import random
import re
import time
import uuid
import zipfile
from pathlib import Path

from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters
from azure.mgmt.storage import StorageManagementClient
from azure.mgmt.storage.models import BlobContainer, StorageAccountCreateParameters, StorageAccountPropertiesCreateParameters
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

## 2. Define Constants

The container name matches the Azure AI Search source lab. The role ID is Azure built-in `Storage Blob Data Contributor`.

In [2]:
CONTAINER_NAME = "documents"
STORAGE_BLOB_DATA_CONTRIBUTOR = "Storage Blob Data Contributor"
STORAGE_BLOB_DATA_CONTRIBUTOR_ROLE_ID = "ba92f5b4-2d11-453d-a403-e96b0029c9fe"
STORAGE_ACCOUNT_NAME_PATTERN = re.compile(r"^[a-z0-9]{3,24}$")

## 3. Find the Workshop Files

This cell locates `workshop/.env`, `documents.zip`, and the local `documents/` folder whether you start the notebook from the repository root or from the notebook folder.

In [3]:
def find_workshop_dir() -> Path:
    start = Path.cwd().resolve()
    candidates = [start / "workshop", start]

    for parent in start.parents:
        candidates.extend([parent / "workshop", parent])

    for candidate in candidates:
        if (candidate / ".env").exists() and (candidate / "lab-03-knowledge-mining").exists():
            return candidate

    raise FileNotFoundError("Could not find workshop/.env. Copy workshop/.env.sample to workshop/.env first.")


workshop_dir = find_workshop_dir()
lab_dir = workshop_dir / "lab-03-knowledge-mining"
env_path = workshop_dir / ".env"
zip_path = lab_dir / "documents.zip"
documents_dir = lab_dir / "documents"

if not zip_path.exists():
    raise FileNotFoundError(f"Documents zip file not found: {zip_path}")

print(f"Workshop directory: {workshop_dir}")
print(f"Environment file: {env_path}")
print(f"Documents zip: {zip_path}")
print(f"Documents folder: {documents_dir}")

Workshop directory: C:\Users\algut\repos\alesaez\content-processing-solution\workshop
Environment file: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\.env
Documents zip: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-03-knowledge-mining\documents.zip
Documents folder: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-03-knowledge-mining\documents


## 4. Add Configuration Helpers

These helpers read values from `workshop/.env`, ignore placeholder values, validate storage account names, and save a generated storage account name back to `.env` for reuse.

In [4]:
def optional_setting(name: str) -> str:
    value = (os.getenv(name) or "").strip()
    return "" if not value or value.startswith("<") else value


def require_setting(name: str) -> str:
    value = optional_setting(name)
    if not value:
        raise ValueError(f"Set {name} in workshop/.env")
    return value


def validate_storage_account_name(account_name: str) -> str:
    if not STORAGE_ACCOUNT_NAME_PATTERN.fullmatch(account_name):
        raise ValueError("STORAGE_ACCOUNT_NAME must be 3 to 24 characters and contain only lowercase letters and numbers.")
    return account_name


def get_storage_account_name() -> str:
    configured_name = optional_setting("STORAGE_ACCOUNT_NAME")
    if configured_name:
        return validate_storage_account_name(configured_name)
    return f"aikm{random.randint(1, 999999):08d}"


def save_env_setting(env_path: Path, name: str, value: str) -> None:
    lines = env_path.read_text(encoding="utf-8").splitlines() if env_path.exists() else []
    setting = f"{name}={value}"

    for index, line in enumerate(lines):
        if line.startswith(f"{name}="):
            lines[index] = setting
            break
    else:
        if lines and lines[-1]:
            lines.append("")
        lines.append(setting)

    env_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

## 5. Load Configuration

This cell loads `.env`, resolves the subscription, resource group, location, and storage account name. Leave `STORAGE_SUBSCRIPTION_ID` blank to infer the subscription from your current Azure credential when exactly one subscription is available. If your account can access multiple subscriptions, set `STORAGE_SUBSCRIPTION_ID` in `workshop/.env`.

In [5]:
import urllib.error
import urllib.request


def get_current_subscription_id(credential: DefaultAzureCredential) -> str:
    try:
        token = credential.get_token("https://management.azure.com/.default").token
        request = urllib.request.Request(
            "https://management.azure.com/subscriptions?api-version=2022-12-01",
            headers={"Authorization": f"Bearer {token}"},
        )
        with urllib.request.urlopen(request) as response:
            subscriptions = json.loads(response.read().decode("utf-8")).get("value", [])
    except urllib.error.HTTPError as exc:
        error_body = exc.read().decode("utf-8", errors="replace")
        raise ValueError(
            "Set STORAGE_SUBSCRIPTION_ID in workshop/.env, or sign in with an Azure identity that can list subscriptions. "
            f"Azure error: {exc.code} {exc.reason}. {error_body}"
        ) from exc
    except Exception as exc:
        raise ValueError(
            "Set STORAGE_SUBSCRIPTION_ID in workshop/.env, or sign in with an Azure identity that can list subscriptions."
        ) from exc

    enabled_subscriptions = [
        subscription
        for subscription in subscriptions
        if str(subscription.get("state", "")).lower() == "enabled"
    ]
    subscriptions = enabled_subscriptions or subscriptions

    if len(subscriptions) == 1:
        subscription_id = (subscriptions[0].get("subscriptionId") or "").strip()
        if subscription_id:
            return subscription_id

    if not subscriptions:
        raise ValueError("Set STORAGE_SUBSCRIPTION_ID in workshop/.env. No Azure subscriptions were found for the current credential.")

    available_subscriptions = "\n".join(
        f"- {subscription.get('displayName', '<unnamed>')}: {subscription.get('subscriptionId', '<missing-id>')}"
        for subscription in subscriptions
    )
    raise ValueError(
        "Multiple Azure subscriptions were found. Set STORAGE_SUBSCRIPTION_ID in workshop/.env to one of these values:\n"
        f"{available_subscriptions}"
    )


load_dotenv(env_path)

credential = DefaultAzureCredential()
subscription_id = optional_setting("STORAGE_SUBSCRIPTION_ID") or optional_setting("AZURE_SUBSCRIPTION_ID")
if not subscription_id:
    subscription_id = get_current_subscription_id(credential)

resource_group = require_setting("STORAGE_RESOURCE_GROUP")
location = os.getenv("STORAGE_LOCATION", "eastus2")
configured_account_name = optional_setting("STORAGE_ACCOUNT_NAME")
account_name = validate_storage_account_name(configured_account_name) if configured_account_name else get_storage_account_name()

print(f"Using subscription: {subscription_id}")
print(f"Storage resource group: {resource_group}")
print(f"Storage location: {location}")
print(f"Storage account name: {account_name}")

Using subscription: cd244f2d-5f05-4dd1-97aa-d130e20bcd15
Storage resource group: TechExcelAOAI
Storage location: eastus2
Storage account name: aiforms00076132


## 6. Create Azure Clients

The credential object uses your current Azure identity. The management clients handle storage account creation and role assignments.

In [6]:
storage_client = StorageManagementClient(credential, subscription_id)
authorization_client = AuthorizationManagementClient(credential, subscription_id)

print("Azure SDK clients created.")

Azure SDK clients created.


## 7. Extract the Local Documents

This cell extracts `documents.zip` only when the `documents/` folder does not already contain PDFs.

In [7]:
def extract_documents(zip_path: Path, documents_dir: Path) -> None:
    if documents_dir.exists() and any(documents_dir.glob("*.pdf")):
        print(f"Using existing extracted documents in {documents_dir}")
        return

    print(f"Extracting {zip_path.name} to {documents_dir}...")
    documents_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(documents_dir)


extract_documents(zip_path, documents_dir)
pdf_files = sorted(documents_dir.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF documents.")
for file_path in pdf_files:
    print(f" - {file_path.name}")

Using existing extracted documents in C:\Users\algut\repos\alesaez\content-processing-solution\workshop\lab-03-knowledge-mining\documents
Found 6 PDF documents.
 - Dubai Brochure.pdf
 - Las Vegas Brochure.pdf
 - London Brochure.pdf
 - Margies Travel Company Info.pdf
 - New York Brochure.pdf
 - San Francisco Brochure.pdf


## 8. Create or Reuse the Storage Account

This cell reuses `STORAGE_ACCOUNT_NAME` if it exists. If the value is blank, it creates a new account and saves the generated name back to `workshop/.env`.

In [8]:
def get_or_create_storage_account(
    storage_client: StorageManagementClient,
    resource_group: str,
    account_name: str,
    location: str,
):
    try:
        storage_account = storage_client.storage_accounts.get_properties(resource_group, account_name)
        print(f"Using existing storage account: {account_name}")
        return storage_account
    except ResourceNotFoundError:
        pass

    print(f"Creating storage account: {account_name}")
    poller = storage_client.storage_accounts.begin_create(
        resource_group,
        account_name,
        StorageAccountCreateParameters(
            sku={"name": "Standard_LRS"},
            kind="StorageV2",
            location=location,
            properties=StorageAccountPropertiesCreateParameters(
                allow_blob_public_access=False,
                enable_https_traffic_only=True,
                minimum_tls_version="TLS1_2",
            ),
        ),
    )
    return poller.result()


def get_public_network_access(storage_account) -> str:
    value = getattr(storage_account, "public_network_access", "")
    return getattr(value, "value", str(value)).lower()


def ensure_blob_container(storage_client: StorageManagementClient, resource_group: str, account_name: str) -> None:
    try:
        storage_client.blob_containers.get(resource_group, account_name, CONTAINER_NAME)
        print(f"Using existing blob container: {CONTAINER_NAME}")
    except ResourceNotFoundError:
        print(f"Creating blob container: {CONTAINER_NAME}")
        storage_client.blob_containers.create(
            resource_group,
            account_name,
            CONTAINER_NAME,
            BlobContainer(public_access="None"),
        )


storage_account = get_or_create_storage_account(storage_client, resource_group, account_name, location)
if not configured_account_name:
    save_env_setting(env_path, "STORAGE_ACCOUNT_NAME", account_name)
    print(f"Saved STORAGE_ACCOUNT_NAME={account_name} to {env_path}")

ensure_blob_container(storage_client, resource_group, account_name)

if get_public_network_access(storage_account) == "disabled":
    raise ValueError(
        f"The {CONTAINER_NAME} container was created, but local document upload cannot continue because "
        f"storage account {account_name} has public network access disabled. Enable public network access for this lab, "
        "or clear STORAGE_ACCOUNT_NAME in workshop/.env so the notebook can create a new lab storage account."
    )

print(f"Storage account resource ID: {storage_account.id}")

Using existing storage account: aiforms00076132
Using existing blob container: documents
Storage account resource ID: /subscriptions/cd244f2d-5f05-4dd1-97aa-d130e20bcd15/resourceGroups/TechExcelAOAI/providers/Microsoft.Storage/storageAccounts/aiforms00076132


## 9. Grant Your User Blob Data Access

This role lets your signed-in user upload the brochure PDFs to the storage container through Microsoft Entra authentication.

In [9]:
def get_current_principal_id(credential: DefaultAzureCredential) -> str:
    token = credential.get_token("https://management.azure.com/.default").token
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    claims = json.loads(base64.urlsafe_b64decode(payload.encode("utf-8")))

    principal_id = (claims.get("oid") or "").strip()
    if not principal_id:
        raise ValueError("Could not determine the current Azure principal object ID from the access token.")
    return principal_id


def grant_current_user_blob_data_contributor(
    authorization_client: AuthorizationManagementClient,
    credential: DefaultAzureCredential,
    storage_account_scope: str,
) -> None:
    principal_id = get_current_principal_id(credential)
    role_definition_id = f"{storage_account_scope}/providers/Microsoft.Authorization/roleDefinitions/{STORAGE_BLOB_DATA_CONTRIBUTOR_ROLE_ID}"
    role_assignment_name = str(uuid.uuid5(uuid.NAMESPACE_URL, f"{storage_account_scope}/{principal_id}/{STORAGE_BLOB_DATA_CONTRIBUTOR_ROLE_ID}"))

    print(f"Granting current user {STORAGE_BLOB_DATA_CONTRIBUTOR} on the storage account...")
    try:
        authorization_client.role_assignments.create(
            storage_account_scope,
            role_assignment_name,
            RoleAssignmentCreateParameters(
                role_definition_id=role_definition_id,
                principal_id=principal_id,
                principal_type="User",
            ),
        )
    except HttpResponseError as exc:
        if exc.error and exc.error.code == "RoleAssignmentExists":
            print(f"Current user already has {STORAGE_BLOB_DATA_CONTRIBUTOR}.")
            return
        raise ValueError(
            f"Could not assign {STORAGE_BLOB_DATA_CONTRIBUTOR}. "
            "Your account must have Owner or User Access Administrator permission on the storage account. "
            f"Azure error: {exc.message}"
        ) from exc


grant_current_user_blob_data_contributor(authorization_client, credential, storage_account.id)

Granting current user Storage Blob Data Contributor on the storage account...


## 10. Upload Documents

This cell uploads every PDF from `workshop/lab-03-knowledge-mining/documents/` to the `documents` container created in the previous step. The retry wrapper handles the short delay Azure RBAC sometimes needs before data-plane permissions take effect.

In [10]:
def upload_documents(blob_service_client: BlobServiceClient, documents_dir: Path) -> None:
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)

    pdf_files = sorted(documents_dir.glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(f"No PDF documents found in {documents_dir}")

    for file_path in pdf_files:
        with file_path.open("rb") as data:
            container_client.upload_blob(name=file_path.name, data=data, overwrite=True)
        print(f"Uploaded: {file_path.name}")


def upload_documents_with_rbac_retry(blob_service_client: BlobServiceClient, documents_dir: Path) -> None:
    retryable_error_codes = {"AuthorizationFailure", "AuthorizationPermissionMismatch"}

    for attempt in range(1, 7):
        try:
            upload_documents(blob_service_client, documents_dir)
            return
        except HttpResponseError as exc:
            if exc.error_code not in retryable_error_codes or attempt == 6:
                raise
            wait_seconds = attempt * 10
            print(f"Waiting {wait_seconds} seconds for role assignment propagation before retrying upload...")
            time.sleep(wait_seconds)


account_url = f"https://{account_name}.blob.core.windows.net"
blob_service_client = BlobServiceClient(account_url=account_url, credential=credential)

print("Uploading documents...")
upload_documents_with_rbac_retry(blob_service_client, documents_dir)
print("Upload complete.")

Uploading documents...
Uploaded: Dubai Brochure.pdf
Uploaded: Las Vegas Brochure.pdf
Uploaded: London Brochure.pdf
Uploaded: Margies Travel Company Info.pdf
Uploaded: New York Brochure.pdf
Uploaded: San Francisco Brochure.pdf
Upload complete.


## 11. Get the Container Details

Use this storage account and container when you connect Azure AI Search to Azure Blob Storage in the **Import data** wizard.

In [11]:
container_url = f"{account_url}/{CONTAINER_NAME}"

print("-------------------------------------")
print(f"Storage account: {account_name}")
print(f"Container: {CONTAINER_NAME}")
print(f"Container URL: {container_url}")
print("Use this storage account and container when importing data into Azure AI Search.")

-------------------------------------
Storage account: aiforms00076132
Container: documents
Container URL: https://aiforms00076132.blob.core.windows.net/documents
Use this storage account and container when importing data into Azure AI Search.
